# E-Commerce Order Data Cleaning

## Project Overview

This project focuses on cleaning and validating a messy e-commerce order dataset.

The dataset contained issues such as duplicate records, inconsistent country names,
mixed date formats, inconsistent price formats, mixed weight units, corrupted text,
and inconsistencies between quantities and order totals.

The goal was to transform the raw data into a clean, consistent dataset ready for
analysis.

## Data Cleaning Summary

The following data-quality issues were identified and addressed:

| Issue | Action Taken |
|---|---|
| Duplicate records | Removed duplicate rows |
| Inconsistent country names | Standardized country names |
| Mixed date formats | Converted dates to a consistent datetime format |
| Prices stored as text | Cleaned currency symbols/codes and converted to numeric |
| Mixed weight units | Standardized weights to grams |
| Corrupted product name | Corrected encoding issue |
| Negative quantities | Investigated and flagged inconsistent statuses |
| Order total differences | Validated totals against applicable discounts |
| Multiple currencies | Retained GBP/EUR because no exchange rate was provided |

### Final Dataset

- **Rows:** 3,000
- **Columns:** 13
- **Duplicate rows:** 0
- **Duplicate order IDs:** 0
- **Missing values:** 0

## Key Findings

During the cleaning and validation process, several important observations were identified:

- 47 duplicate records were removed.
- Country names appeared in multiple formats and were standardized.
- The `order_date` column contained mixed date formats and was standardized.
- Some prices contained currency symbols/codes and were converted to numeric values.
- Weight values were stored using two different scales and were standardized to grams.
- The product name `CafÃ© Latte Mug` contained an encoding issue and was corrected.
- 102 orders contained negative quantities, indicating returns/refunds.
- 88 negative-quantity orders were marked as `completed`, which was flagged as a potential data-quality issue.
- Order totals were checked against calculated totals and common discount rates rather than being blindly overwritten.
- Both GBP and EUR were retained because no exchange rate was provided.

The final dataset contains **3,000 rows and 13 columns** with no duplicate rows or duplicate order IDs.

In [1]:
import pandas as pd

df = pd.read_csv("raw/01-ecommerce-orders.csv")

print(df.head())
print(df.shape)
print(df.columns)

     order_id  order_date            customer_email  country  channel  \
0  ORD-101805  2026-03-19  customer0457@example.com   France   direct   
1  ORD-102690  19/05/2026  customer0814@example.com   France   social   
2  ORD-101454  2026-06-14  customer0858@example.com       UK    email   
3  ORD-100450  2026-01-08  customer0116@example.com       UK  organic   
4  ORD-101501  2026-05-20  customer1397@example.com  Ireland    email   

  product_sku       product_name  quantity unit_price  order_total   weight  \
0     FR-9902     Trekking Poles         1      45.00         36.0  520.000   
1     FR-2210    Trail Softshell         1      71.10         71.1    0.520   
2     FR-1120    Fell Runner Cap         1      45.00         45.0    0.085   
3     FR-5567  Merino Base Layer         1      71.10         71.1  190.000   
4     FR-8890  Summit Down Gilet         1      21.60         21.6    0.380   

  currency     status  
0      EUR  completed  
1      GBP  completed  
2      GBP  co

In [2]:
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 3047 entries, 0 to 3046
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        3047 non-null   str    
 1   order_date      3047 non-null   str    
 2   customer_email  3047 non-null   str    
 3   country         3047 non-null   str    
 4   channel         3047 non-null   str    
 5   product_sku     3047 non-null   str    
 6   product_name    3047 non-null   str    
 7   quantity        3047 non-null   int64  
 8   unit_price      3047 non-null   str    
 9   order_total     3047 non-null   float64
 10  weight          3047 non-null   float64
 11  currency        3047 non-null   str    
 12  status          3047 non-null   str    
dtypes: float64(2), int64(1), str(10)
memory usage: 596.2 KB
None

Missing values:
order_id          0
order_date        0
customer_email    0
country           0
channel           0
product_sku       0
product_name      0
quanti

In [3]:
print("Duplicate order IDs:", df["order_id"].duplicated().sum())

print("\nCountries:")
print(df["country"].value_counts())

print("\nCurrencies:")
print(df["currency"].value_counts())

print("\nChannels:")
print(df["channel"].value_counts())

print("\nStatuses:")
print(df["status"].value_counts())

Duplicate order IDs: 47

Countries:
country
uk                320
United Kingdom    320
 UK               311
UK                310
France            306
Ireland           304
ireland           298
gb                298
GB                293
Germany           287
Name: count, dtype: int64

Currencies:
currency
GBP    2397
EUR     650
Name: count, dtype: int64

Channels:
channel
paid_search    654
organic        623
direct         610
email          585
social         575
Name: count, dtype: int64

Statuses:
status
completed    2291
refunded      756
Name: count, dtype: int64


In [4]:
duplicates = df[df["order_id"].duplicated(keep=False)]

print(duplicates.sort_values("order_id").to_string(index=False))

  order_id order_date           customer_email        country     channel product_sku       product_name  quantity unit_price  order_total  weight currency    status
ORD-100050 07/02/2026 customer0853@example.com         France paid_search     FR-9902     Trekking Poles         1      71.10        71.10 520.000      GBP  refunded
ORD-100050 07/02/2026 customer0853@example.com         France paid_search     FR-9902     Trekking Poles         1      71.10        71.10 520.000      GBP  refunded
ORD-100066 03/15/2026 customer0604@example.com             gb paid_search     FR-4471      Alpine Jacket         3     149.00       447.00 640.000      EUR completed
ORD-100066 03/15/2026 customer0604@example.com             gb paid_search     FR-4471      Alpine Jacket         3     149.00       447.00 640.000      EUR completed
ORD-100096 2026-04-13 customer0189@example.com             UK     organic     FR-5567  Merino Base Layer         1      89.00        89.00 190.000      EUR completed
ORD-

In [5]:
print("Exact duplicate rows:", df.duplicated().sum())

Exact duplicate rows: 47


In [6]:
exact_duplicates = df[df.duplicated(keep=False)]

print(exact_duplicates.to_string(index=False))

  order_id order_date           customer_email        country     channel product_sku       product_name  quantity unit_price  order_total  weight currency    status
ORD-100485 2026-06-14 customer1056@example.com            uk        email     FR-5567  Merino Base Layer         1      45.00        45.00   0.190      GBP completed
ORD-102794 2026-05-21 customer0480@example.com        ireland paid_search     FR-7781 Storm Overtrousers         2     129.00       258.00   0.410      GBP completed
ORD-101429 2026-05-26 customer0368@example.com             UK     organic     FR-7781 Storm Overtrousers         3      24.00        72.00   0.410      GBP  refunded
ORD-101570 2026-04-07 customer0843@example.com             UK paid_search     FR-2210    Trail Softshell         2      11.25        22.50 520.000      GBP completed
ORD-100485 2026-06-14 customer1056@example.com            uk        email     FR-5567  Merino Base Layer         1      45.00        45.00   0.190      GBP completed
ORD-

In [7]:
print("Exact duplicate rows:", df.duplicated().sum())


Exact duplicate rows: 47


In [8]:
df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))

Rows after removing duplicates: 3000


In [9]:
df["country"] = df["country"].str.strip().str.lower()

country_map = {
    "uk": "United Kingdom",
    "gb": "United Kingdom",
    "united kingdom": "United Kingdom",
    "ireland": "Ireland",
    "france": "France",
    "germany": "Germany"
}

df["country"] = df["country"].replace(country_map)

print(df["country"].value_counts())

country
United Kingdom    1823
Ireland            593
France             303
Germany            281
Name: count, dtype: int64


In [10]:
print(df["order_date"].head(20).to_string(index=False))

2026-03-19
19/05/2026
2026-06-14
2026-01-08
2026-05-20
2026-03-14
2026-04-25
2026-05-20
02/04/2026
16/02/2026
21/04/2026
01/07/2026
02/27/2026
21/02/2026
2026-07-05
2026-01-11
19/06/2026
07/06/2026
2026-04-23
2026-06-14


In [11]:
print("Date type:", df["order_date"].dtype)

Date type: str


In [12]:
df["order_date"] = pd.to_datetime(
    df["order_date"],
    format="mixed",
    errors="coerce",
    dayfirst=True
)

print(df["order_date"].head(20))
print("\nMissing dates after conversion:", df["order_date"].isna().sum())
print("\nData type:", df["order_date"].dtype)

0    2026-03-19
1    2026-05-19
2    2026-06-14
3    2026-08-01
4    2026-05-20
5    2026-03-14
6    2026-04-25
7    2026-05-20
8    2026-04-02
9    2026-02-16
10   2026-04-21
11   2026-07-01
12   2026-02-27
13   2026-02-21
14   2026-05-07
15   2026-11-01
16   2026-06-19
17   2026-06-07
18   2026-04-23
19   2026-06-14
Name: order_date, dtype: datetime64[us]

Missing dates after conversion: 0

Data type: datetime64[us]


In [13]:
df["order_date"].isna().sum()

np.int64(0)

In [14]:
print(df["unit_price"].head(20).to_string(index=False))
print("\nData type:", df["unit_price"].dtype)

 45.00
 71.10
 45.00
 71.10
 21.60
 45.00
 85.50
134.10
 24.00
 95.00
 45.00
129.00
116.10
129.00
 45.00
 71.10
116.10
 89.00
149.00
 45.00

Data type: str


In [15]:
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

print("Data type:", df["unit_price"].dtype)
print("Missing prices:", df["unit_price"].isna().sum())

Data type: float64
Missing prices: 228


In [16]:
raw_df = pd.read_csv("raw/01-ecommerce-orders.csv")

print(raw_df.loc[
    pd.to_numeric(raw_df["unit_price"], errors="coerce").isna(),
    "unit_price"
].to_string(index=False))

 GBP45.00
   £80.10
 GBP12.50
 GBP80.10
   £85.50
  £116.10
GBP116.10
   £80.10
   £24.00
   £71.10
GBP129.00
 GBP95.00
   £40.50
 GBP95.00
 GBP12.50
GBP134.10
   £11.25
   £79.00
 GBP89.00
 GBP24.00
 GBP71.10
  £134.10
   £89.00
  £149.00
   £21.60
   £11.25
GBP116.10
   £79.00
GBP134.10
   £21.60
 GBP45.00
  £149.00
   £21.60
 GBP71.10
   £12.50
   £40.50
   £79.00
  £129.00
   £24.00
 GBP89.00
  £149.00
   £85.50
  £149.00
 GBP45.00
   £89.00
 GBP24.00
GBP149.00
   £89.00
GBP134.10
   £24.00
   £79.00
 GBP21.60
 GBP89.00
 GBP89.00
 GBP11.25
   £85.50
   £40.50
 GBP95.00
 GBP79.00
   £71.10
   £79.00
   £89.00
 GBP79.00
 GBP40.50
  £134.10
  £149.00
   £11.25
   £12.50
   £71.10
GBP134.10
  £149.00
 GBP11.25
GBP149.00
   £71.10
   £40.50
 GBP45.00
  £129.00
 GBP79.00
   £79.00
   £12.50
 GBP12.50
 GBP24.00
  £129.00
   £12.50
  £129.00
 GBP12.50
GBP129.00
   £12.50
 GBP40.50
  £149.00
   £85.50
   £11.25
 GBP95.00
   £85.50
   £12.50
 GBP24.00
  £129.00
  £149.00
 GBP89.00
   £79.00


In [17]:
df = pd.read_csv("raw/01-ecommerce-orders.csv")

# Remove exact duplicate rows
df = df.drop_duplicates()

# Standardize countries
df["country"] = df["country"].str.strip().str.lower()

country_map = {
    "uk": "United Kingdom",
    "gb": "United Kingdom",
    "united kingdom": "United Kingdom",
    "ireland": "Ireland",
    "france": "France",
    "germany": "Germany"
}

df["country"] = df["country"].replace(country_map)

# Convert mixed dates
df["order_date"] = pd.to_datetime(
    df["order_date"],
    format="mixed",
    errors="coerce",
    dayfirst=True
)

# Clean unit price
df["unit_price"] = (
    df["unit_price"]
    .str.replace("GBP", "", regex=False)
    .str.replace("£", "", regex=False)
    .str.strip()
)

df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

print("Rows:", len(df))
print("Missing prices:", df["unit_price"].isna().sum())

Rows: 3000
Missing prices: 0


In [18]:
df["calculated_total"] = df["quantity"] * df["unit_price"]

df["difference"] = df["order_total"] - df["calculated_total"]

print(df["difference"].describe())

count    3000.000000
mean       -1.402223
std         7.034566
min       -89.400000
25%         0.000000
50%         0.000000
75%         0.000000
max        40.230000
Name: difference, dtype: float64


In [19]:
mismatched = df[df["difference"] != 0]

print("Mismatched orders:", len(mismatched))
print("\nPercentage:", len(mismatched) / len(df) * 100, "%")

Mismatched orders: 300

Percentage: 10.0 %


In [20]:
print(mismatched[
    ["order_id", "quantity", "unit_price", "order_total", "calculated_total", "difference"]
].head(20).to_string(index=False))

  order_id  quantity  unit_price  order_total  calculated_total    difference
ORD-101805         1       45.00        36.00              45.0 -9.000000e+00
ORD-102773         3      116.10       348.30             348.3  5.684342e-14
ORD-100053         1       45.00        36.00              45.0 -9.000000e+00
ORD-101237         2       95.00       171.00             190.0 -1.900000e+01
ORD-102523         1      129.00       103.20             129.0 -2.580000e+01
ORD-100798         3       21.60        64.80              64.8 -1.421085e-14
ORD-101972         3      116.10       348.30             348.3  5.684342e-14
ORD-100431         1       24.00        20.40              24.0 -3.600000e+00
ORD-102810         3      129.00       348.30             387.0 -3.870000e+01
ORD-100683         3      116.10       348.30             348.3  5.684342e-14
ORD-101024         3      116.10       348.30             348.3  5.684342e-14
ORD-101379         3      129.00       328.95             387.0 

In [21]:
mismatched = df[abs(df["difference"]) > 0.01].copy()

mismatched["discount_percent"] = (
    (mismatched["calculated_total"] - mismatched["order_total"])
    / mismatched["calculated_total"]
    * 100
)

print(mismatched["discount_percent"].round(2).value_counts().head(15))

discount_percent
20.00    77
15.00    69
10.00    48
15.01     6
15.04     3
10.04     2
15.02     2
14.99     1
Name: count, dtype: int64


In [22]:
mismatched["discount_percent_rounded"] = (
    mismatched["discount_percent"].round()
)

print(
    mismatched["discount_percent_rounded"]
    .value_counts()
    .sort_index()
)

discount_percent_rounded
10.0    50
15.0    81
20.0    77
Name: count, dtype: int64


In [23]:
expected_totals = df["quantity"] * df["unit_price"]

discount_rates = [0, 0.10, 0.15, 0.20]

valid = pd.Series(False, index=df.index)

for rate in discount_rates:
    valid |= (
        (df["order_total"] - expected_totals * (1 - rate)).abs() <= 0.01
    )

print("Orders matching a valid total:", valid.sum())
print("Orders still unexplained:", (~valid).sum())

Orders matching a valid total: 3000
Orders still unexplained: 0


In [24]:
print(df["weight"].describe())

print("\nSample weights:")
print(df["weight"].head(30).to_string(index=False))

count    3000.000000
mean      227.517313
std       231.105207
min         0.085000
25%         0.410000
50%       190.000000
75%       410.000000
max       640.000000
Name: weight, dtype: float64

Sample weights:
520.000
  0.520
  0.085
190.000
  0.380
 85.000
  0.085
  0.085
190.000
190.000
  0.410
410.000
340.000
640.000
  0.520
 85.000
  0.410
  0.340
  0.640
  0.190
190.000
380.000
 85.000
340.000
410.000
  0.190
340.000
  0.380
  0.380
520.000


In [25]:
print(df["weight"].value_counts().sort_index().head(30))

weight
0.085      154
0.190      161
0.340      149
0.380      156
0.410      148
0.520      303
0.640      172
85.000     228
190.000    191
340.000    217
380.000    217
410.000    236
520.000    451
640.000    217
Name: count, dtype: int64


In [26]:
print(df[["product_name", "weight"]].sort_values("weight").head(20).to_string(index=False))

   product_name  weight
Fell Runner Cap   0.085
Fell Runner Cap   0.085
Fell Runner Cap   0.085
Fell Runner Cap   0.085
Fell Runner Cap   0.085
Fell Runner Cap   0.085
Fell Runner Cap   0.085
Fell Runner Cap   0.085
Fell Runner Cap   0.085
Fell Runner Cap   0.085
Fell Runner Cap   0.085
Fell Runner Cap   0.085
Fell Runner Cap   0.085
Fell Runner Cap   0.085
Fell Runner Cap   0.085
Fell Runner Cap   0.085
Fell Runner Cap   0.085
Fell Runner Cap   0.085
Fell Runner Cap   0.085
Fell Runner Cap   0.085


In [27]:
print(
    df.groupby("product_name")["weight"]
      .unique()
      .to_string()
)

product_name
Alpine Jacket         [640.0, 0.64]
CafÃ© Latte Mug       [340.0, 0.34]
Fell Runner Cap       [0.085, 85.0]
Merino Base Layer     [190.0, 0.19]
Storm Overtrousers    [0.41, 410.0]
Summit Down Gilet     [0.38, 380.0]
Trail Softshell       [0.52, 520.0]
Trekking Poles        [520.0, 0.52]


In [28]:
df.loc[df["weight"] < 1, "weight"] *= 1000

print(df["weight"].value_counts().sort_index())

weight
85.0     382
190.0    352
340.0    366
380.0    373
410.0    384
520.0    754
640.0    389
Name: count, dtype: int64


In [29]:
print(df.groupby("product_name")["weight"].unique().to_string())

product_name
Alpine Jacket         [640.0]
CafÃ© Latte Mug       [340.0]
Fell Runner Cap        [85.0]
Merino Base Layer     [190.0]
Storm Overtrousers    [410.0]
Summit Down Gilet     [380.0]
Trail Softshell       [520.0]
Trekking Poles        [520.0]


In [30]:
print(df["product_name"].value_counts().to_string())

product_name
Alpine Jacket         389
Storm Overtrousers    384
Trekking Poles        382
Fell Runner Cap       382
Summit Down Gilet     373
Trail Softshell       372
CafÃ© Latte Mug       366
Merino Base Layer     352


In [31]:
df["product_name"] = df["product_name"].replace(
    "CafÃ© Latte Mug",
    "Café Latte Mug"
)

print(df["product_name"].value_counts())

product_name
Alpine Jacket         389
Storm Overtrousers    384
Trekking Poles        382
Fell Runner Cap       382
Summit Down Gilet     373
Trail Softshell       372
Café Latte Mug        366
Merino Base Layer     352
Name: count, dtype: int64


In [32]:
print(df["quantity"].value_counts().sort_index())

quantity
-3      15
-2      37
-1      50
 1    1448
 2     966
 3     484
Name: count, dtype: int64


In [33]:
print(
    pd.crosstab(df["quantity"] < 0, df["status"])
)

status    completed  refunded
quantity                     
False          2164       734
True             88        14


In [34]:
suspicious = df[
    (df["quantity"] < 0) &
    (df["status"] == "completed")
]

print(
    suspicious[
        ["order_id", "product_name", "quantity", "unit_price",
         "order_total", "status"]
    ].head(20).to_string(index=False)
)

  order_id       product_name  quantity  unit_price  order_total    status
ORD-101676 Storm Overtrousers        -1      129.00      -129.00 completed
ORD-102075     Café Latte Mug        -2       45.00       -90.00 completed
ORD-102055  Merino Base Layer        -1      149.00      -149.00 completed
ORD-102439  Summit Down Gilet        -2      129.00      -258.00 completed
ORD-102481      Alpine Jacket        -1       12.50       -12.50 completed
ORD-101426      Alpine Jacket        -1       80.10       -80.10 completed
ORD-102830      Alpine Jacket        -1       45.00       -45.00 completed
ORD-102598  Merino Base Layer        -2      129.00      -258.00 completed
ORD-100897      Alpine Jacket        -1       45.00       -45.00 completed
ORD-101439      Alpine Jacket        -3       45.00      -135.00 completed
ORD-102878      Alpine Jacket        -2       12.50       -25.00 completed
ORD-101636    Fell Runner Cap        -2       11.25       -22.50 completed
ORD-101502      Alpine Ja

In [35]:
status_issue = df[
    (df["quantity"] < 0) &
    (df["status"] == "completed")
]

print("Orders with negative quantity but completed status:", len(status_issue))

Orders with negative quantity but completed status: 88


In [36]:
print(df.groupby("currency")[["unit_price", "order_total"]].agg(["min", "max", "mean"]))

         unit_price                   order_total                   
                min    max       mean         min    max        mean
currency                                                            
EUR           11.25  149.0  73.883934      -348.3  447.0  117.438103
GBP           11.25  149.0  74.628175      -298.0  447.0  113.527295


In [37]:
print(df["currency"].value_counts())

currency
GBP    2362
EUR     638
Name: count, dtype: int64


In [38]:
print(df.groupby("currency")["order_total"].sum())

currency
EUR     74925.51
GBP    268151.47
Name: order_total, dtype: float64


In [39]:
print(pd.crosstab(df["country"], df["currency"]))

currency        EUR   GBP
country                  
France           57   246
Germany          62   219
Ireland         127   466
United Kingdom  392  1431


In [40]:
print(df.isnull().sum())

order_id            0
order_date          0
customer_email      0
country             0
channel             0
product_sku         0
product_name        0
quantity            0
unit_price          0
order_total         0
weight              0
currency            0
status              0
calculated_total    0
difference          0
dtype: int64


In [41]:
print(df.dtypes)

order_id                       str
order_date          datetime64[us]
customer_email                 str
country                        str
channel                        str
product_sku                    str
product_name                   str
quantity                     int64
unit_price                 float64
order_total                float64
weight                     float64
currency                       str
status                         str
calculated_total           float64
difference                 float64
dtype: object


In [42]:
df = df.drop(columns=["calculated_total", "difference"])

print(df.columns)

Index(['order_id', 'order_date', 'customer_email', 'country', 'channel',
       'product_sku', 'product_name', 'quantity', 'unit_price', 'order_total',
       'weight', 'currency', 'status'],
      dtype='str')


In [43]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate order IDs:", df["order_id"].duplicated().sum())

Duplicate rows: 0
Duplicate order IDs: 0


In [44]:
df.to_csv("output/cleaned_ecommerce_orders.csv", index=False)

print("Cleaned file saved successfully!")

Cleaned file saved successfully!


In [45]:
print(df.shape)
print(df.head())

(3000, 13)
     order_id order_date            customer_email         country  channel  \
0  ORD-101805 2026-03-19  customer0457@example.com          France   direct   
1  ORD-102690 2026-05-19  customer0814@example.com          France   social   
2  ORD-101454 2026-06-14  customer0858@example.com  United Kingdom    email   
3  ORD-100450 2026-08-01  customer0116@example.com  United Kingdom  organic   
4  ORD-101501 2026-05-20  customer1397@example.com         Ireland    email   

  product_sku       product_name  quantity  unit_price  order_total  weight  \
0     FR-9902     Trekking Poles         1        45.0         36.0   520.0   
1     FR-2210    Trail Softshell         1        71.1         71.1   520.0   
2     FR-1120    Fell Runner Cap         1        45.0         45.0    85.0   
3     FR-5567  Merino Base Layer         1        71.1         71.1   190.0   
4     FR-8890  Summit Down Gilet         1        21.6         21.6   380.0   

  currency     status  
0      EUR  com